阶段1：构造模型

在票价确定的前提下，预测各航段的客流量（已完成）。

阶段2：解析过程

根据客流信息分配舱位，以确定总收益。
客舱分为两类：短途类（AB或BC，称为S类）和长途类（AC，称为L类），总容量为C。

根据是否满座进行分类：

若不满座（S+L≤C），则按S、L和C的预测比例分配舱位，以实现收益最大化。
若满座（S+L>C），则需根据销售量变化预测进行取舍，并作如下判断：
若票价AB+BC>AC，则优先将舱位拆分为AB和BC段销售，S的值取min(AB, BC)，其余舱位分配给L。
若票价AB+BC<AC，则优先分配舱位给AC段，剩余舱位分配给AB和BC段。

阶段3：求总收益最大值

以票价为自变量，总收益为因变量，绘制总收益曲线，并求出总收益的最大值。

## 加载CSV文件

In [1]:
import pandas as pd
import numpy as np

In [2]:
# # 读取CSV文件
# # /Users/zhanyu/code/data-hh/海航系销售结果数据
# df = pd.read_csv('../../data-hh/海航系销售结果数据/mid_flt_allmkt_segment_result_sale_tmp_202410221659.csv')

# # 显示前两行数据以确保正确加载
# print(df.shape)
# print(df.head(5))

In [3]:
import pandas as pd
import glob
import os

# 指定目录路径
directory_path = '../../data-hh/海航系销售结果数据/'

# 使用 glob 匹配所有 CSV 文件
csv_files = glob.glob(os.path.join(directory_path, "*.csv"))

# 检查匹配到的 CSV 文件列表
print("找到的 CSV 文件:")
print(csv_files)

# 读取所有 CSV 文件并合并为一个 DataFrame
dataframes = []  # 用于存储每个 CSV 文件的 DataFrame
for file in csv_files:
    print(f"正在读取文件: {file}")
    df = pd.read_csv(file)
    dataframes.append(df)

# 合并所有 DataFrame（可根据需要选择拼接方式）
df = pd.concat(dataframes, ignore_index=True)

# 显示合并后的数据形状和前几行
print("合并后的数据形状:", df.shape)
print(df.head())


找到的 CSV 文件:
['../../data-hh/海航系销售结果数据/hh_result_23-10-12.csv', '../../data-hh/海航系销售结果数据/hh_result_24-3-4.csv', '../../data-hh/海航系销售结果数据/hh_result_24-5-6.csv', '../../data-hh/海航系销售结果数据/hh_result_24-1-2.csv', '../../data-hh/海航系销售结果数据/hh_result_23-6-7.csv', '../../data-hh/海航系销售结果数据/hh_result_23-4-5.csv', '../../data-hh/海航系销售结果数据/hh_result_23-8-9.csv', '../../data-hh/海航系销售结果数据/hh_result_23-1-3.csv']
正在读取文件: ../../data-hh/海航系销售结果数据/hh_result_23-10-12.csv
正在读取文件: ../../data-hh/海航系销售结果数据/hh_result_24-3-4.csv
正在读取文件: ../../data-hh/海航系销售结果数据/hh_result_24-5-6.csv
正在读取文件: ../../data-hh/海航系销售结果数据/hh_result_24-1-2.csv
正在读取文件: ../../data-hh/海航系销售结果数据/hh_result_23-6-7.csv
正在读取文件: ../../data-hh/海航系销售结果数据/hh_result_23-4-5.csv
正在读取文件: ../../data-hh/海航系销售结果数据/hh_result_23-8-9.csv
正在读取文件: ../../data-hh/海航系销售结果数据/hh_result_23-1-3.csv
合并后的数据形状: (6546417, 15)
     flt_date                    segment        flt_no bd_type  dep_time  cap  \
0  2023-10-01  KNqX4/Q5Noc=-HexFWXqbb8I=  KgJrsp7Jd78=      窄体  12:15:

## 拆分 flt_date 为 year, month, day, weekday

In [4]:
# 将 'flt_date' 列转换为 datetime 格式
df['flt_date'] = pd.to_datetime(df['flt_date'])

# 提取年、月、日和星期几
df['year'] = df['flt_date'].dt.year
df['month'] = df['flt_date'].dt.month
df['day'] = df['flt_date'].dt.day
df['weekday'] = df['flt_date'].dt.weekday  # 0 = Monday, 6 = Sunday

# 删除原始的 'flt_date' 字段
df = df.drop(columns=['flt_date'])

# 查看结果
print(df.head(5))

                     segment        flt_no bd_type  dep_time  cap aircraft  \
0  KNqX4/Q5Noc=-HexFWXqbb8I=  KgJrsp7Jd78=      窄体  12:15:00  132      319   
1  AKQNtuL5r6Q=-Mv6HkAiSLUk=  P9IRwar34h0=      窄体  20:10:00  189      321   
2  n465JzB8Rrw=-N4hmDZN/CJQ=  mJitm0UDfM4=      窄体  16:45:00  132      319   
3  N4hmDZN/CJQ=-n465JzB8Rrw=  jXr97M1wpn4=      窄体  19:40:00  132      319   
4  5t+HPO9Mu/w=-X5e5r3CS4OA=  izjfHOxAho4=      窄体  13:05:00  132      319   

   legs  leg_no  duration   tkt_rev  pax             a             b    c  \
0     1       1      1.07   30880.0   25  KNqX4/Q5Noc=  HexFWXqbb8I=  NaN   
1     1       1      1.38  110730.0  151  AKQNtuL5r6Q=  Mv6HkAiSLUk=  NaN   
2     1       1      1.57   32090.0   38  n465JzB8Rrw=  N4hmDZN/CJQ=  NaN   
3     1       1      1.58  139520.0  109  N4hmDZN/CJQ=  n465JzB8Rrw=  NaN   
4     1       1      1.80  125180.0  124  5t+HPO9Mu/w=  X5e5r3CS4OA=  NaN   

   year  month  day  weekday  
0  2023     10    1        6  
1  202

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6546417 entries, 0 to 6546416
Data columns (total 18 columns):
 #   Column    Dtype  
---  ------    -----  
 0   segment   object 
 1   flt_no    object 
 2   bd_type   object 
 3   dep_time  object 
 4   cap       int64  
 5   aircraft  object 
 6   legs      int64  
 7   leg_no    int64  
 8   duration  float64
 9   tkt_rev   float64
 10  pax       int64  
 11  a         object 
 12  b         object 
 13  c         object 
 14  year      int32  
 15  month     int32  
 16  day       int32  
 17  weekday   int32  
dtypes: float64(2), int32(4), int64(4), object(8)
memory usage: 799.1+ MB


## 拆分 dep_time 为 hour, minute, second

In [6]:
# 拆分 'dep_time' 字段
df[['hour', 'minute', 'second']] = df['dep_time'].str.split(':', expand=True)

# 将拆分的结果转换为整数
df['hour'] = df['hour'].astype(int)
df['minute'] = df['minute'].astype(int)
df['second'] = df['second'].astype(int)

# 删除原始的 'dep_time' 字段
df = df.drop(columns=['dep_time'])

# 查看结果
print(df.head(5))

                     segment        flt_no bd_type  cap aircraft  legs  \
0  KNqX4/Q5Noc=-HexFWXqbb8I=  KgJrsp7Jd78=      窄体  132      319     1   
1  AKQNtuL5r6Q=-Mv6HkAiSLUk=  P9IRwar34h0=      窄体  189      321     1   
2  n465JzB8Rrw=-N4hmDZN/CJQ=  mJitm0UDfM4=      窄体  132      319     1   
3  N4hmDZN/CJQ=-n465JzB8Rrw=  jXr97M1wpn4=      窄体  132      319     1   
4  5t+HPO9Mu/w=-X5e5r3CS4OA=  izjfHOxAho4=      窄体  132      319     1   

   leg_no  duration   tkt_rev  pax             a             b    c  year  \
0       1      1.07   30880.0   25  KNqX4/Q5Noc=  HexFWXqbb8I=  NaN  2023   
1       1      1.38  110730.0  151  AKQNtuL5r6Q=  Mv6HkAiSLUk=  NaN  2023   
2       1      1.57   32090.0   38  n465JzB8Rrw=  N4hmDZN/CJQ=  NaN  2023   
3       1      1.58  139520.0  109  N4hmDZN/CJQ=  n465JzB8Rrw=  NaN  2023   
4       1      1.80  125180.0  124  5t+HPO9Mu/w=  X5e5r3CS4OA=  NaN  2023   

   month  day  weekday  hour  minute  second  
0     10    1        6    12      15       0 

## 拆分 segment 为 from 和 to

In [7]:
# 拆分 'segment' 字段为 'from' 和 'to'
df[['from', 'to']] = df['segment'].str.split('-', expand=True)

# 删除原始的 'segment' 字段
df = df.drop(columns=['segment'])

# 查看结果
print(df.head(10))

         flt_no bd_type  cap aircraft  legs  leg_no  duration   tkt_rev  pax  \
0  KgJrsp7Jd78=      窄体  132      319     1       1      1.07   30880.0   25   
1  P9IRwar34h0=      窄体  189      321     1       1      1.38  110730.0  151   
2  mJitm0UDfM4=      窄体  132      319     1       1      1.57   32090.0   38   
3  jXr97M1wpn4=      窄体  132      319     1       1      1.58  139520.0  109   
4  izjfHOxAho4=      窄体  132      319     1       1      1.80  125180.0  124   
5  nKALH9OikCM=      窄体  132      319     1       1      2.13   40390.0  107   
6  ya5APB9GgSc=      窄体  164      320     1       1      2.63   87190.0  127   
7  cWEn0916Iho=      窄体  164      320     1       1      2.08   38280.0   55   
8  2Ak9P6oD/xk=      窄体  132      319     1       1      2.07   19510.0   27   
9  l0Cpo80mBi8=      窄体  132      319     1       1      2.40  135140.0   87   

              a  ...    c  year  month  day  weekday  hour  minute  second  \
0  KNqX4/Q5Noc=  ...  NaN  2023     10   

## 删除pax异常的行

In [8]:
# # 统计 'pax' 字段中缺失值的行数
# missing_pax = df[df['pax'].isna()]

# # 统计 'pax' 字段中0值的行数
# zero_pax = df[df['pax'] == 0]

# # 输出统计结果
# num_missing = missing_pax.shape[0]
# num_zero = zero_pax.shape[0]

# print(df.shape[0])
# print(f"缺失值的行数: {num_missing}")
# print(f"为0的行数: {num_zero}")

In [9]:
# # 删除 'pax' 字段为缺失值或为0的行
# df = df.dropna(subset=['pax'])  # 删除pax列中的缺失值行
# df = df[df['pax'] != 0]  # 删除pax列中为0的行

# # 查看删除后的DataFrame行数
# num_rows_after_cleanup = df.shape[0]
# print(f"删除缺失值或为0的行后，DataFrame一共有 {num_rows_after_cleanup} 行")

In [10]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

def remove_outliers(data, col):
    """删除指定列中的异常值"""
    Q1 = data[col].quantile(0.25)  # 第1四分位数
    Q3 = data[col].quantile(0.75)  # 第3四分位数
    IQR = Q3 - Q1  # 四分位距
    lower_bound = Q1 - 1.5 * IQR  # 下限
    upper_bound = Q3 + 1.5 * IQR  # 上限
    
    # 筛选出范围内的数据
    filtered_data = data[(data[col] >= lower_bound) & (data[col] <= upper_bound)]
    return filtered_data

def plot(data, col):
    """绘制指定列的分布图和箱型图"""
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6))
    sns.histplot(data[col], kde=True, ax=ax1)  # 替换 distplot 为 histplot
    sns.boxplot(data[col], ax=ax2)
    plt.tight_layout()
    plt.show()

# # 绘制过滤后的数据分布图和箱型图
# plot(df, 'pax')
# # 示例数据框
# # 假设 df 是你的数据框
# # 删除 pax 列中的异常值
# df = remove_outliers(df, 'pax')

# # 绘制过滤后的数据分布图和箱型图
# plot(df, 'pax')


In [11]:
# 删除 pax 小于 20 的行
df = df[df['pax'] >= 20]

## 处理aircraft

In [12]:
# 查看 'cap' 列中每个元素的数据类型
print(df['aircraft'].apply(type).value_counts())

aircraft
<class 'str'>      6094378
<class 'float'>         85
Name: count, dtype: int64


In [13]:
# 统计 'pax' 字段中缺失值的行数
missing_pax = df[df['aircraft'].isna()]

# 统计 'pax' 字段中0值的行数
zero_pax = df[df['aircraft'] == 0]

# 输出统计结果
num_missing = missing_pax.shape[0]
num_zero = zero_pax.shape[0]

print(df.shape[0])
print(f"缺失值的行数: {num_missing}")
print(f"为0的行数: {num_zero}")

6094463
缺失值的行数: 85
为0的行数: 0


In [14]:
# 将 cap 列转换为 Pandas 的 Int64 类型，这种类型支持 NaN 值
# 将 'aircraft' 列转换为字符串类型
# 用 'Unknown' 填充 NaN 值
df['aircraft'] = df['aircraft'].fillna('Unknown')
df['aircraft'] = df['aircraft'].astype(str)

In [15]:
# 查看 'cap' 列中每个元素的数据类型
print(df['aircraft'].apply(type).value_counts())

aircraft
<class 'str'>    6094463
Name: count, dtype: int64


## 处理cap

In [16]:
# 查看 'cap' 列中每个元素的数据类型
print(df['cap'].apply(type).value_counts())

cap
<class 'int'>    6094463
Name: count, dtype: int64


In [17]:
# 统计 'pax' 字段中缺失值的行数
missing_pax = df[df['cap'].isna()]

# 统计 'pax' 字段中0值的行数
zero_pax = df[df['cap'] == 0]

# 输出统计结果
num_missing = missing_pax.shape[0]
num_zero = zero_pax.shape[0]

print(df.shape[0])
print(f"缺失值的行数: {num_missing}")
print(f"为0的行数: {num_zero}")

# 将 'cap' 字段中为0的值修改为 NaN
df.loc[df['cap'] == 0, 'cap'] = np.nan

# 将 'cap' 字段转换为字符串类型
df['cap'] = df['cap'].astype(str)



6094463
缺失值的行数: 0
为0的行数: 615673


In [18]:
# 查看 'cap' 列中每个元素的数据类型
print(df['cap'].apply(type).value_counts())
print(df['aircraft'].apply(type).value_counts())

cap
<class 'str'>    6094463
Name: count, dtype: int64
aircraft
<class 'str'>    6094463
Name: count, dtype: int64


## 处理duration（待做）

观察数据后发现只有甩飞航线的ac段的duration缺失，这个可以通过ab+bc得到

In [19]:
# 将 'duration' 列中值为 0 的部分替换为 NaN
df.loc[df['duration'] == 0, 'duration'] = np.nan

## 处理tkt_rev

有一些异常行的tkt_rev为0，pax却不为0

In [20]:
# 统计 'unit_price' 列中为 0 的行数
zero_count = (df['tkt_rev'] == 0).sum()

# 统计 'unit_price' 列中为 NaN 的行数
nan_count = df['tkt_rev'].isna().sum()

# 输出结果
print(f"tkt_rev 列中为 0 的行数: {zero_count}")
print(f"tkt_rev 列中为 NaN 的行数: {nan_count}")

tkt_rev 列中为 0 的行数: 228
tkt_rev 列中为 NaN 的行数: 0


In [21]:
# df = df[df['tkt_rev'] != 0]

# 计算单价 'unit_price'，当 pax=0 时设置为 NaN
df['unit_price'] = np.where(df['pax'] == 0, 
                           np.nan, 
                           df['tkt_rev'] / df['pax'])

# # 删除 'tkt_rev' 列
# df = df.drop(columns=['tkt_rev'])

In [22]:
# 统计 'pax' 字段异常值
missing_pax = df[df['pax'].isna()]
zero_pax = df[df['pax'] == 0]

# 统计 'tkt_rev' 字段异常值
missing_tkt_rev = df[df['tkt_rev'].isna()]
zero_tkt_rev = df[df['tkt_rev'] == 0]

# 输出统计结果
print(f"数据集总行数: {df.shape[0]}")

print("\npax异常值统计：")
print(f"缺失值的行数: {len(missing_pax)}")
print(f"为0的行数: {len(zero_pax)}")
print(f"pax异常行占总行数的比例: {(len(missing_pax) + len(zero_pax))/df.shape[0]*100:.2f}%")

print("\ntkt_rev异常值统计：")
print(f"缺失值的行数: {len(missing_tkt_rev)}")
print(f"为0的行数: {len(zero_tkt_rev)}")
print(f"tkt_rev异常行占总行数的比例: {(len(missing_tkt_rev) + len(zero_tkt_rev))/df.shape[0]*100:.2f}%")

# 显示一些特殊情况的示例
print("\npax为0但tkt_rev不为0的示例行：")
print(df[(df['pax'] == 0) & (df['tkt_rev'] > 0)].head())

print("\ntkt_rev为0但pax不为0的示例行：")
print(df[(df['tkt_rev'] == 0) & (df['pax'] > 0)].head())

数据集总行数: 6094463

pax异常值统计：
缺失值的行数: 0
为0的行数: 0
pax异常行占总行数的比例: 0.00%

tkt_rev异常值统计：
缺失值的行数: 0
为0的行数: 228
tkt_rev异常行占总行数的比例: 0.00%

pax为0但tkt_rev不为0的示例行：
Empty DataFrame
Columns: [flt_no, bd_type, cap, aircraft, legs, leg_no, duration, tkt_rev, pax, a, b, c, year, month, day, weekday, hour, minute, second, from, to, unit_price]
Index: []

[0 rows x 22 columns]

tkt_rev为0但pax不为0的示例行：
             flt_no bd_type    cap aircraft  legs  leg_no  duration  tkt_rev  \
31659  FXXebMExCwc=      窄体  180.0      320     3       2      2.85      0.0   
31663  dtSxpfzyeQo=      窄体  180.0      320     3       1      2.85      0.0   
53418  Z63nbZSal+I=      窄体  200.0      321     1       1      3.73      0.0   
54570  BGIkmk9ahPw=      窄体  200.0      321     1       1      3.70      0.0   
56388  wsulA8baimc=      支线   90.0      ARJ     3       1      1.85      0.0   

       pax             a  ...  year month  day  weekday  hour  minute  second  \
31659  176  Y50ETJ8qYSE=  ...  2023    10    3        1

In [23]:
# 首先显示删除前的行数
print(f"删除前的行数: {df.shape[0]}")

# 找出需要删除的异常行
abnormal_rows_1 = df[(df['pax'] == 0) & (df['tkt_rev'] > 0)]
abnormal_rows_2 = df[(df['tkt_rev'] == 0) & (df['pax'] > 0)]

print(f"\n要删除的异常行数：")
print(f"pax为0但tkt_rev不为0的行数: {len(abnormal_rows_1)}")
print(f"tkt_rev为0但pax不为0的行数: {len(abnormal_rows_2)}")

# 删除异常行
df = df[~((df['pax'] == 0) & (df['tkt_rev'] > 0))]  # 删除 pax=0 但 tkt_rev>0 的行
df = df[~((df['tkt_rev'] == 0) & (df['pax'] > 0))]  # 删除 tkt_rev=0 但 pax>0 的行

# 显示删除后的行数
print(f"\n删除后的行数: {df.shape[0]}")
print(f"共删除了 {len(abnormal_rows_1) + len(abnormal_rows_2)} 行异常数据")

# 验证是否还存在异常行
remaining_abnormal_1 = df[(df['pax'] == 0) & (df['tkt_rev'] > 0)]
remaining_abnormal_2 = df[(df['tkt_rev'] == 0) & (df['pax'] > 0)]
print(f"\n验证结果：")
print(f"剩余 pax为0但tkt_rev不为0 的行数: {len(remaining_abnormal_1)}")
print(f"剩余 tkt_rev为0但pax不为0 的行数: {len(remaining_abnormal_2)}")

删除前的行数: 6094463

要删除的异常行数：
pax为0但tkt_rev不为0的行数: 0
tkt_rev为0但pax不为0的行数: 228

删除后的行数: 6094235
共删除了 228 行异常数据

验证结果：
剩余 pax为0但tkt_rev不为0 的行数: 0
剩余 tkt_rev为0但pax不为0 的行数: 0


## 保存文件

In [24]:
# 保存处理后的数据到新的CSV文件
df.to_csv('../../data-hh/my/hh_result/result_all.csv', index=False)

In [25]:
# 筛选数据
filtered_df = df[
    (df['flt_no'] == '/O2MVlG1lP0=') & 
    (df['year'] == 2024) & 
    (df['month'] == 1) & 
    (df['day'] == 1)
]

# 显示筛选结果
print("筛选结果的形状:", filtered_df.shape)
print("\n筛选结果:")
print(filtered_df)

筛选结果的形状: (3, 22)

筛选结果:
               flt_no bd_type    cap aircraft  legs  leg_no  duration  \
2495556  /O2MVlG1lP0=      窄体  194.0      321     3       1      2.42   
2496755  /O2MVlG1lP0=      窄体    nan      321     3       3       NaN   
2496756  /O2MVlG1lP0=      窄体  194.0      321     3       2      2.72   

          tkt_rev  pax             a  ...  year month  day  weekday  hour  \
2495556  100166.0  102  xIqe+QQIQXQ=  ...  2024     1    1        0     7   
2496755   80735.0   56  xIqe+QQIQXQ=  ...  2024     1    1        0     7   
2496756   44778.0   94  xIqe+QQIQXQ=  ...  2024     1    1        0    10   

         minute  second          from            to   unit_price  
2495556       0       0  xIqe+QQIQXQ=  AfKac4FzdXs=   982.019608  
2496755       0       0  xIqe+QQIQXQ=  So/c8CkA/Xs=  1441.696429  
2496756      55       0  AfKac4FzdXs=  So/c8CkA/Xs=   476.361702  

[3 rows x 22 columns]


## 统计航段价格信息

In [26]:
# 读取预处理后的数据
df = pd.read_csv('../../data-hh/my/hh_result/result_all.csv')

In [27]:
# 计算每对城市间直航航班的价格统计信息
price_stats = df.groupby(['from', 'to']).agg({
    'unit_price': ['mean', 'std', 'count']
}).round(2)

# 重命名列
price_stats.columns = ['平均价格', '价格标准差', '航班数量']

# 按平均价格降序排序
price_stats = price_stats.sort_values('平均价格', ascending=False)

# 显示结果
print("每对城市间直航航班的价格统计：")
print(price_stats)

# 可选：保存结果到CSV文件
price_stats.to_csv('../../data-hh/my/hh_result/route_price_stats.csv')

# 输出一些基本统计信息
print("\n基本统计信息：")
print(f"航线总数：{len(price_stats)}")
print(f"最高平均票价航线：{price_stats['平均价格'].max():.2f}")
print(f"最低平均票价航线：{price_stats['平均价格'].min():.2f}")
print(f"最大标准差航线：{price_stats['价格标准差'].max():.2f}")

每对城市间直航航班的价格统计：
                              平均价格    价格标准差  航班数量
from         to                                  
yypkiQCX5lk= nlFV4N9et20=  4630.00      NaN     1
nlFV4N9et20= yypkiQCX5lk=  4594.62   360.17     6
t0yIIzgLFLs= 0J4jz5aCatU=  4567.94  3671.76     6
So/c8CkA/Xs= ZCvDTbZuS8Q=  4500.00      NaN     1
Y50ETJ8qYSE= EIsXRbFUH34=  4460.00      NaN     1
...                            ...      ...   ...
iX2Pe2pxiqQ= wbf1sxPhwNM=    99.00     4.27    35
wbf1sxPhwNM= iX2Pe2pxiqQ=    98.17     3.65    25
2jt80AsNp3c= Y50ETJ8qYSE=    92.50    51.04    18
Y50ETJ8qYSE= 2jt80AsNp3c=    60.86    33.51    13
LIrnUnlANso= 2jt80AsNp3c=    26.48      NaN     1

[7910 rows x 3 columns]

基本统计信息：
航线总数：7910
最高平均票价航线：4630.00
最低平均票价航线：26.48
最大标准差航线：3671.76
